# Libraries

In [1]:
from pathlib import Path
import os
import sys
import numpy as np
import torch
from torch.utils.data import ConcatDataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.metrics import f1_score

In [2]:
PROJECT_ROOT = Path.cwd()
PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import agents
from dataloaders.base import RafDB_perm

# Setup

In [ ]:
WEIGHT_PATH = Path(r"")

In [4]:
BATCH_SIZE = 64
NUM_WORKERS = 0
GPUID = [0]

CATEGORY = "gender"
MODEL_TYPE = "resnet"
MODEL_NAME = "TVResNet18_pretrained_freeze"
AGENT_NAME = "ProposedFramework"
SKIP_UNSURE_FOR_GENDER = True
SKIP_UNSURE = SKIP_UNSURE_FOR_GENDER if CATEGORY == "gender" else False

In [5]:
def patch_resnet_no_download():
    import models.resnet as repo_resnet
    repo_resnet.TVResNet18_pretrained_freeze = lambda: repo_resnet.TVResNet("resnet18", pretrained=False, freeze_initial=True)

def load_state_dict(path):
    state = torch.load(path, map_location="cpu")
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    if all(k.startswith("module.") for k in state.keys()):
        state = {k[len("module."):]: v for k, v in state.items()}
    return state

patch_resnet_no_download()

agent_config = {
    "lr": 0.001,
    "momentum": 0.0,
    "weight_decay": 0.0,
    "schedule": [1],
    "model_type": MODEL_TYPE,
    "model_name": MODEL_NAME,
    "model_weights": None,
    "out_dim": {"All": 7},
    "optimizer": "Adam",
    "print_freq": 0,
    "gpuid": GPUID,
    "reg_coef": 0.0,
    "prune_amount": 0.0,
    "target_acc": 0.85,
    "alpha": 0.05,
    "init_prune_rate": 0.95,
    "inc_prune_rate": 0.50,
    "prune_retrain_epochs": 0,
    "lambda_bias": 1.0,
}

agent_factory = getattr(agents.customization, AGENT_NAME)
agent = agent_factory(agent_config)
state_dict = load_state_dict(WEIGHT_PATH)
agent.model.load_state_dict(state_dict, strict=True)
agent.eval()

C:\Users\gassa\AppData\Local\Temp\ipykernel_18392\293544804.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(path, map_location="cpu")


ProposedFramework(
  (model): TVResNet(
    (model): ResNet(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU(inplace=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
        (1): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): Ba

# Load Data

In [6]:
_, val_datasets, _ = RafDB_perm(CATEGORY, train_aug=False, skip_unsure=SKIP_UNSURE)
task_names = sorted(val_datasets.keys(), key=int)
val_all = ConcatDataset([val_datasets[name] for name in task_names])
val_loader = DataLoader(val_all, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print("tasks:", task_names)
print("validation samples:", len(val_all))

Loading cached attributes...


Reading classes: 100%|██████████| 15339/15339 [00:00<00:00, 589923.15line/s]
c:\Users\gassa\Kaiwen\UM\2025_2026\Semester_1\Project\.GitCode\Benchmarkwith_RAF-DB\dataloaders\wrapper.py:23: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on G

tasks: ['1', '2']
validation samples: 2869


# Eval

In [38]:
def evaluate(agent, loader):
    device = next(agent.model.parameters()).device
    y_true, y_pred = [], []
    group_total = {}
    group_correct = {}

    agent.eval()
    with torch.no_grad():
        for inputs, targets, tasks in loader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            logits = agent.predict(inputs)["All"]
            preds = logits.argmax(dim=1)

            y_true.extend(targets.cpu().tolist())
            y_pred.extend(preds.cpu().tolist())

            correct = preds.eq(targets).cpu().tolist()
            for task, is_correct in zip(tasks, correct):
                task = str(task)
                group_total[task] = group_total.get(task, 0) + 1
                group_correct[task] = group_correct.get(task, 0) + int(is_correct)

    overall_acc = np.mean(np.array(y_true) == np.array(y_pred))
    group_acc = {task: group_correct[task] / group_total[task] for task in group_total}
    group_values = list(group_acc.values())
    acc_gap = max(group_values) - min(group_values)
    fairness_ratio = min(group_values) / max(group_values) if max(group_values) > 0 else 0.0

    return overall_acc, group_total, group_correct, group_acc, acc_gap, fairness_ratio

In [39]:
overall_acc, group_total, group_correct, group_acc, acc_gap, fairness_ratio = evaluate(agent, val_loader)

nonzero_params = agent.count_parameter()
total_params = sum(p.numel() for p in agent.model.parameters())
sparsity = 1.0 - (nonzero_params / total_params)

In [40]:
print(f"Overall accuracy:   {overall_acc * 100:.2f}%")
print(f"Fairness Score:     {fairness_ratio:.4f}")
print(f"Params:             {nonzero_params:,} / {total_params:,}")
print(f"Pruned:             {sparsity * 100:.2f}%")

print("\nGroup accuracy:")
for task in sorted(group_total.keys(), key=int):
    print(
        f"  Task {task}: {group_acc[task] * 100:6.2f}% "
        f"({group_correct[task]}/{group_total[task]})"
    )

Overall accuracy:   74.83%
Fairness Score:     0.9818
Params:             1,319,564 / 11,180,103
Pruned:             88.20%

Group accuracy:
  Task 1:  74.06% (925/1249)
  Task 2:  75.43% (1222/1620)


# Inference

In [34]:
EMOTION_LABELS = [
    "Surprise",
    "Fear",
    "Disgust",
    "Happiness",
    "Sadness",
    "Anger",
    "Neutral",
]

IMAGE_PATHS = [
    # "RafDB/basic/Image/aligned/test_0001_aligned.jpg",
    'notebooks/images/Male_Happiness.PNG',
    'notebooks/images/Male_Anger.PNG',
    'notebooks/images/Female_Happiness.PNG',
    'notebooks/images/Female_Anger.PNG',
]

infer_transform = transforms.Compose([
    transforms.Resize((100, 100)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def resolve_path(path):
    path = Path(path)
    return path if path.is_absolute() else PROJECT_ROOT / path

def get_inference_paths():
    paths = [resolve_path(path) for path in IMAGE_PATHS]
    missing = [path for path in paths if not path.exists()]

    if missing:
        raise FileNotFoundError("Missing image file(s): " + ", ".join(str(path) for path in missing))

    return paths

def predict_image(path):
    image = Image.open(path).convert("RGB")
    tensor = infer_transform(image).unsqueeze(0)
    device = next(agent.model.parameters()).device

    agent.eval()
    with torch.no_grad():
        logits = agent.predict(tensor.to(device))["All"]
        probabilities = torch.softmax(logits, dim=1)[0].cpu()

    pred_label = int(probabilities.argmax())
    confidence = float(probabilities[pred_label])
    return pred_label, confidence, probabilities


In [37]:
for image_path in get_inference_paths():
    pred_label, confidence, probabilities = predict_image(image_path)
    print("Image:", image_path)
    print("Predicted label:", EMOTION_LABELS[pred_label])
    print(f"Confidence: {confidence:.4f}")
    print("Class probabilities:")
    for label, prob in zip(EMOTION_LABELS, probabilities.tolist()):
        print(f"  {label:>10}: {prob:.4f}")
    print()

Image: c:\Users\gassa\Kaiwen\UM\2025_2026\Semester_1\Project\.GitCode\Benchmarkwith_RAF-DB\notebooks\images\Male_Happiness.PNG
Predicted label: Happiness
Confidence: 0.3526
Class probabilities:
    Surprise: 0.0799
        Fear: 0.0519
     Disgust: 0.0912
   Happiness: 0.3526
     Sadness: 0.1945
       Anger: 0.0511
     Neutral: 0.1788

Image: c:\Users\gassa\Kaiwen\UM\2025_2026\Semester_1\Project\.GitCode\Benchmarkwith_RAF-DB\notebooks\images\Male_Anger.PNG
Predicted label: Anger
Confidence: 0.9956
Class probabilities:
    Surprise: 0.0003
        Fear: 0.0020
     Disgust: 0.0011
   Happiness: 0.0005
     Sadness: 0.0005
       Anger: 0.9956
     Neutral: 0.0000

Image: c:\Users\gassa\Kaiwen\UM\2025_2026\Semester_1\Project\.GitCode\Benchmarkwith_RAF-DB\notebooks\images\Female_Happiness.PNG
Predicted label: Happiness
Confidence: 0.2787
Class probabilities:
    Surprise: 0.1061
        Fear: 0.0384
     Disgust: 0.1037
   Happiness: 0.2787
     Sadness: 0.1820
       Anger: 0.0564
  